In [ ]:
!pip install pandas matplotlib seaborn --quiet
!pip install pandas tqdm --quiet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from tqdm import tqdm
from google.colab import drive
from google.colab import files
drive.mount('/content/drive')
start_time = time.time()
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/clean_data.csv")
def infer_age(row):
  return "senior" if row["SeniorCitizen"] == 1 else "adult"
def build_profile(row):
    # Gender & age
    if "gender" in row:
        gender_val = str(row["gender"])
        if gender_val.lower() in ["0", "m", "male"]:
            gender = "Male"
        elif gender_val.lower() in ["1", "f", "female"]:
            gender = "Female"
        else:
            gender = "Customer"
    else:
        gender = "Customer"

    # Age
    age_desc = "senior" if "SeniorCitizen" in row and row["SeniorCitizen"] == 1 else "adult"

    # Relationship / dependents
    partner = "married or living with a partner" if str(row.get("Partner", "")).lower() in ["yes", "1"] else "single"
    dependents = "with children or dependents" if str(row.get("Dependents", "")).lower() in ["yes", "1"] else "without children"

    # Tenure group
    tenure_desc = None
    for t in ["tenure_<12", "tenure_12-24", "tenure_24-36", "tenure_36-48", "tenure_48-60", "tenure_>60"]:
        if t in row.index and row[t] == 1:
            tenure_desc = t.split("_")[1].replace("-", " to ") + " months"
            break
    if not tenure_desc and "tenure" in row:
        t = row["tenure"]
        if t < 12: tenure_desc = "<12 months"
        elif t < 24: tenure_desc = "12–24 months"
        elif t < 36: tenure_desc = "24–36 months"
        elif t < 48: tenure_desc = "36–48 months"
        elif t < 60: tenure_desc = "48–60 months"
        else: tenure_desc = ">60 months"

    # Internet & services
    internet = None
    if "InternetService_DSL" in row and row["InternetService_DSL"] == 1: internet = "DSL"
    elif "InternetService_Fiber optic" in row and row["InternetService_Fiber optic"] == 1: internet = "Fiber optic"
    elif "InternetService_No" in row and row["InternetService_No"] == 1: internet = "no internet service"

    tech_support = "has technical support" if str(row.get("TechSupport", "")).lower() in ["yes", "1"] else "lacks tech support"
    paperless = "uses paperless billing" if str(row.get("PaperlessBilling", "")).lower() in ["yes", "1"] else "prefers mailed bills"
    autopay = "uses automatic payment" if str(row.get("AutomaticPayment", "")).lower() in ["yes", "1"] else "does not use automatic payment"

    # Spending
    monthly = f"${row['MonthlyCharges']:.0f}/month" if "MonthlyCharges" in row and not pd.isnull(row["MonthlyCharges"]) else "unknown monthly spending"
    total = f"totaling ${row['TotalCharges']:.0f}" if "TotalCharges" in row and not pd.isnull(row["TotalCharges"]) else ""

    # Churn
    churn_status = "is at risk of churn" if row.get("Churn") == 1 else "appears loyal"

    # Combine into readable text
    profile = (
        f"{gender} {age_desc} customer, {partner} and {dependents}. "
        f"They have been with the company for about {tenure_desc}, use {internet} internet, "
        f"{tech_support}, and {paperless}. The customer {autopay}, "
        f"pays around {monthly} {total}, and {churn_status}."
    )
    return profile
tqdm.pandas()
df["Customer_Profile"] = df.progress_apply(build_profile, axis=1)
# End timing
end_time = time.time()

# Calculate elapsed time
elapsed = end_time - start_time
print(f"⏱️ Benchmark: Process completed in {elapsed:.2f} seconds")
df[["Customer_Profile"]].to_csv("customer_profiles.csv", index=False)
print("\n✅ Saved 'customer_profiles.csv' with one profile per customer.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 7010/7010 [00:00<00:00, 15546.82it/s]


⏱️ Benchmark: Process completed in 1.40 seconds

✅ Saved 'customer_profiles.csv' with one profile per customer.
